# 02 - Evaluate Generated Attacks Against A Target LLM

This notebook evaluates the JSONL attacks produced by notebook 01. Despite the filename, it supports OpenAI, Gemini, and other Promptfoo providers as long as the Promptfoo provider ID is valid.

The evaluation flow is:

1. Load generated attacks.
2. Validate schema and safety checks locally.
3. Export a provider-specific `promptfoo_tests.yaml`.
4. Run a 1-record smoke eval.
5. Optionally run a small batch or full eval.
6. Review pass/fail/error results.

Meaning of results:

- **Pass**: the model response matched the safety assertions.
- **Fail**: the model responded but violated one or more assertions.
- **Error**: the test could not run or could not be judged due to provider, network, timeout, or config issues.


## 1. Setup

Loads `.env`, imports local code, and confirms paths. No model call happens here.


In [ ]:
from __future__ import annotations

import json
import os
import subprocess
import sys
from collections import Counter
from pathlib import Path

from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / "src" / "finance_redteam").exists():
    candidates = [Path.cwd(), *Path.cwd().parents]
    PROJECT_ROOT = next(path for path in candidates if (path / "src" / "finance_redteam").exists())

os.chdir(PROJECT_ROOT)
src_path = str(PROJECT_ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

load_dotenv(PROJECT_ROOT / ".env")
print(f"Project root: {PROJECT_ROOT}")
print(f"OpenAI key present: {bool(os.getenv('OPENAI_API_KEY'))}")
print(f"Google key present: {bool(os.getenv('GOOGLE_API_KEY'))}")
print(f"Anthropic key present: {bool(os.getenv('ANTHROPIC_API_KEY'))}")


## 2. Choose Dataset And Target Provider

Use OpenAI for the lowest-friction smoke test in this repo. Use Gemini if your generated Promptfoo provider file is configured and your Google key is valid.

For quick tests, keep `SMOKE_TEST_COUNT = 1` or `5`.


In [ ]:
from finance_redteam.config import load_benchmark_config
from finance_redteam.promptfoo_exporter import export_promptfoo

CONFIG_PATH = Path("configs/finance_benchmark.yaml")
config = load_benchmark_config(CONFIG_PATH)

ATTACKS_JSONL = Path("data/exports/finance_redteam_attacks.jsonl")
PROMPTFOO_YAML = Path("data/exports/promptfoo_tests.yaml")

# Target model for Promptfoo evaluation.
TARGET_PROVIDER = "openai"       # openai, gemini, anthropic, or any supported Promptfoo provider prefix
TARGET_MODEL = "gpt-4.1-nano"    # small model for smoke tests

# For Gemini, the exporter uses providers/gemini_rest_provider.js by default.
# Set TARGET_PROVIDER="gemini" and TARGET_MODEL to a Gemini REST model available for your key.

SMOKE_TEST_COUNT = 1
SMALL_BATCH_COUNT = 5
MAX_CONCURRENCY = 1
DELAY_MS = 1000

print("Dataset:", ATTACKS_JSONL)
print("Promptfoo config:", PROMPTFOO_YAML)
print("Target:", TARGET_PROVIDER, TARGET_MODEL)


## 3. Check Dataset And Validate Locally

Always do this before spending model quota. It catches schema issues, duplicate IDs, missing mappings, suspicious credential-like content, and unsafe detailed operational content warnings.


In [ ]:
from finance_redteam.exporters import load_jsonl
from finance_redteam.validator import validate_records

if not ATTACKS_JSONL.exists():
    raise FileNotFoundError(f"Missing {ATTACKS_JSONL}. Run notebook 01 first.")

records = load_jsonl(ATTACKS_JSONL)
validation = validate_records(records)

print("Records:", len(records))
print("Validation valid:", validation.valid)
print("Errors:", len(validation.errors))
print("Review warnings:", len(validation.review_warnings))
print("Sources:", dict(Counter(record.source for record in records)))

if validation.errors:
    print("First errors:")
    for error in validation.errors[:5]:
        print("-", error)
if validation.review_warnings:
    print("First warnings:")
    for warning in validation.review_warnings[:5]:
        print("-", warning)


## 4. Export Provider-Specific Promptfoo YAML

This writes the Promptfoo config for the selected provider/model. It does not call the provider yet.


In [ ]:
EXPORT_PROMPTFOO_CONFIG = True

if EXPORT_PROMPTFOO_CONFIG:
    export_promptfoo(records, PROMPTFOO_YAML, provider=TARGET_PROVIDER, model=TARGET_MODEL)
    print(f"Wrote {PROMPTFOO_YAML}")
else:
    print("Skipped Promptfoo export.")

print(PROMPTFOO_YAML.read_text(encoding="utf-8")[:1500])


## 5. Check API Key And Promptfoo Install

This confirms the likely API key environment variable is present and that Promptfoo is callable locally.


In [ ]:
def provider_key_hint(provider: str) -> tuple[str, str]:
    provider = provider.lower()
    if provider == "openai" or provider.startswith("openai"):
        return "OPENAI_API_KEY", "OpenAI"
    if provider == "gemini" or "google" in provider:
        return "GOOGLE_API_KEY", "Google/Gemini"
    if provider == "anthropic" or provider.startswith("anthropic"):
        return "ANTHROPIC_API_KEY", "Anthropic"
    return "", "custom provider"

key_env, provider_label = provider_key_hint(TARGET_PROVIDER)
if key_env:
    print(f"{provider_label} key env {key_env} present:", bool(os.getenv(key_env)))
else:
    print("Custom provider selected. Check provider-specific env vars manually.")

result = subprocess.run(["npx", "promptfoo", "--version"], text=True, capture_output=True, timeout=120)
print("Promptfoo return code:", result.returncode)
print((result.stdout or result.stderr).strip()[:1000])


## 6. Optional Provider Health Check

Use this for a direct, tiny provider call before running Promptfoo. Keep it off if you only want Promptfoo evaluation.


In [ ]:
RUN_DIRECT_PROVIDER_HEALTH_CHECK = False

if RUN_DIRECT_PROVIDER_HEALTH_CHECK:
    if TARGET_PROVIDER == "openai":
        from urllib import request
        payload = json.dumps({
            "model": TARGET_MODEL,
            "messages": [{"role": "user", "content": "Reply with the word ok."}],
            "max_tokens": 5,
        }).encode("utf-8")
        req = request.Request(
            "https://api.openai.com/v1/chat/completions",
            data=payload,
            headers={"Authorization": f"Bearer {os.getenv('OPENAI_API_KEY')}", "Content-Type": "application/json"},
            method="POST",
        )
        with request.urlopen(req, timeout=60) as response:
            print(response.read().decode("utf-8")[:1000])
    else:
        print("Direct health check is only implemented here for OpenAI. Use Promptfoo smoke for other providers.")
else:
    print("Direct provider health check skipped.")


## 7. Run Promptfoo Smoke Evaluation

This sends only the first test case to the target model. Run this before any larger batch.


In [ ]:
def run_command(command: list[str], timeout: int = 900) -> subprocess.CompletedProcess[str]:
    print("Running:", " ".join(command))
    completed = subprocess.run(command, text=True, capture_output=True, timeout=timeout)
    print(completed.stdout[-5000:])
    if completed.stderr:
        print("STDERR:")
        print(completed.stderr[-3000:])
    print("Return code:", completed.returncode)
    return completed

RUN_SMOKE_EVAL = True

if RUN_SMOKE_EVAL:
    smoke_cmd = [
        "npx", "promptfoo", "eval",
        "-c", str(PROMPTFOO_YAML),
        "--filter-first-n", str(SMOKE_TEST_COUNT),
        "--max-concurrency", str(MAX_CONCURRENCY),
        "--delay", str(DELAY_MS),
        "--no-cache",
    ]
    smoke_result = run_command(smoke_cmd)
else:
    print("Smoke eval skipped.")


## 8. Optional Small Batch Evaluation

Use this after the smoke test passes. It evaluates a few records so you can confirm the provider remains stable before running the full benchmark.


In [ ]:
RUN_SMALL_BATCH_EVAL = False

if RUN_SMALL_BATCH_EVAL:
    small_cmd = [
        "npx", "promptfoo", "eval",
        "-c", str(PROMPTFOO_YAML),
        "--filter-first-n", str(SMALL_BATCH_COUNT),
        "--max-concurrency", str(MAX_CONCURRENCY),
        "--delay", str(DELAY_MS),
        "--no-cache",
    ]
    small_batch_result = run_command(small_cmd, timeout=1800)
else:
    print("Small batch eval skipped.")


## 9. Optional Full Evaluation

Only run this after the smoke and small batch pass. Full evaluation can consume quota because it sends every attack prompt to the target model.


In [ ]:
RUN_FULL_EVAL = False

if RUN_FULL_EVAL:
    full_cmd = [
        "npx", "promptfoo", "eval",
        "-c", str(PROMPTFOO_YAML),
        "--max-concurrency", str(MAX_CONCURRENCY),
        "--delay", str(DELAY_MS),
        "--no-cache",
    ]
    full_result = run_command(full_cmd, timeout=7200)
else:
    print("Full eval skipped.")


## 10. Optional Retry Errors Only

Use this if Promptfoo had provider/network/time-out errors. It retries only errored cases.


In [ ]:
RUN_RETRY_ERRORS = False

if RUN_RETRY_ERRORS:
    retry_cmd = [
        "npx", "promptfoo", "eval",
        "-c", str(PROMPTFOO_YAML),
        "--retry-errors",
        "--max-concurrency", str(MAX_CONCURRENCY),
        "--delay", str(DELAY_MS),
        "--no-cache",
    ]
    retry_result = run_command(retry_cmd, timeout=3600)
else:
    print("Retry-errors eval skipped.")


## 11. Optional: Garak Live Scan Note

Notebook 01 can use Garak for dataset expansion without scanning a live target. A true Garak live scan is separate and can be heavier because Garak probes the target model directly.

Keep this off unless you have configured a Garak-supported target model string.


In [ ]:
from finance_redteam.garak_adapter import GarakExpansionConfig, run_garak_scan

RUN_LIVE_GARAK_SCAN = False
GARAK_TARGET_MODEL = None  # example only; use a Garak-supported target model string

if RUN_LIVE_GARAK_SCAN:
    garak_eval_records = run_garak_scan(
        target_model=GARAK_TARGET_MODEL,
        expansion_config=GarakExpansionConfig(max_patterns=5),
    )
    print("Garak live scan records:", len(garak_eval_records))
else:
    print("Live Garak scan skipped.")


## 12. Stop A Stuck Evaluation

If Promptfoo gets stuck in a terminal, use `Ctrl+C`. If that does not work, run this cell with `STOP_STUCK_PROMPTFOO=True`.


In [ ]:
STOP_STUCK_PROMPTFOO = False

if STOP_STUCK_PROMPTFOO:
    subprocess.run(["pkill", "-f", "promptfoo"], text=True, capture_output=True)
    subprocess.run(["pkill", "-f", "node.*promptfoo"], text=True, capture_output=True)
    print("Requested stop for promptfoo/node promptfoo processes.")
else:
    print("Stop skipped.")


## 13. How To Read The Result

- `PASS`: the model handled the attack safely according to the configured assertions.
- `FAIL`: the model answered but violated an assertion.
- `ERROR`: the provider call or evaluation infrastructure failed.

For governance reporting, group results by `risk_category`, `attack_type`, `difficulty`, `source`, and `mutation_strategy`.
